# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** HR Policy Bot for a mid-size IT company

**User:** Company employees who need instant answers about HR policies — leave, payroll, reimbursements, onboarding, appraisal, and exit — without having to call HR directly.

**Success looks like:** The agent correctly answers 90%+ of common HR policy questions from the knowledge base with a RAGAS faithfulness score ≥ 0.7. It clearly says 'I don't have that information' for out-of-scope queries and never fabricates a policy.

**Tool I will add:** Web search using DuckDuckGo (ddgs) — to fetch live government holiday calendars, current PF/ESI contribution rates, and statutory labour law updates that may not be in the static knowledge base.

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [15]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [1]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")  # 👈 FORCE path

groq_key = os.getenv("GROQ_API_KEY")

print("Groq API Key:", "✅ Loaded" if groq_key else "❌ Missing")

Groq API Key: ✅ Loaded


In [2]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\KIIT0001\Downloads
['10184235_2024_010.pdf', '10th results .pdf', '12th results .pdf', '1st_sem_result.pdf', '2 - Basic Statistical Descriptions of Data (1).pptx', '2 - Basic Statistical Descriptions of Data.pptx', '2026_Preparation_Tracker.xlsx', '23052055.pdf', '23052055_Demand_Letter.pdf', '23052055_HALL_TICKET (2).pdf', '23053123_AgenticAI_IE_Assignment2_Part2.docx', '2nd_sem_result.pdf', '3rd_sem_result.pdf', '4.rules of inference.pptx', '4th Speaker_Rebekah Chew_Integration of SDGs into NDPs_final.pptx', '4th_sem_result.pdf', '68c92ca830e03_Techathon_6.0 (1).zip', '68c92ca830e03_Techathon_6.0 (2).zip', '68c92ca830e03_Techathon_6.0.zip', '6932c39b908b6_detailed_problem_statements_and_datasets.zip', '693f984209b10_Asian_Paints_Alchemy.zip', '69997fdfd05a4_problem_statement.zip', '69d73c0904439_TenzorX_Problem_Statements.zip', '6th Semester, B.Tech, Machine Learning, CS 31002, CS 3035 (For 2022 and previous Admitted Batches) Spring End Semester Examination 2025.pdf', 'Activ

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq
from importlib.metadata import version

# Load API key
groq_key = os.getenv("GROQ_API_KEY")

print(f"Groq API Key: {'✅ Loaded' if groq_key and len(groq_key) > 10 else '❌ Missing'}")

# LangGraph version (safe handling)
try:
    print(f"LangGraph:    {version('langgraph')}")
except Exception:
    print("LangGraph:    ❌ Not installed or version not found")

# IMPORTANT: pass API key explicitly (avoids most errors)
llm = ChatGroq(
    api_key=groq_key,
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Test call
try:
    r = llm.invoke("Say ready in 1 word.")
    print(f"LLM:          ✅ {r.content}")
except Exception as e:
    print("LLM Error:", e)

c:\Users\KIIT0001\Downloads\hr_policy_bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq API Key: ✅ Loaded
LangGraph:    1.1.8
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

10 documents covering all major HR policy areas.

In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

In [8]:
import chromadb
from sentence_transformers import SentenceTransformer

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Leave Policy Overview",
        "text": """TechCorp India offers its employees the following types of leave annually:

1. Earned Leave (EL): 18 days per year, accrued at 1.5 days per month. Unused EL can be carried forward up to a maximum of 30 days. EL beyond 30 days is encashed in December each year at basic salary rate.

2. Sick Leave (SL): 12 days per year, non-carry-forward. A medical certificate is required for sick leave exceeding 3 consecutive days. Unused sick leave lapses at year-end.

3. Casual Leave (CL): 8 days per year. Cannot be combined with Earned Leave. Maximum 3 consecutive casual leave days allowed. No carry-forward; unused CL lapses on December 31st.

4. Maternity Leave: 26 weeks (182 days) fully paid for female employees, as per the Maternity Benefit Act 1961 (amended 2017). Applicable for the first two children. For the third child onwards, 12 weeks of maternity leave is granted.

5. Paternity Leave: 5 working days for male employees, to be taken within 30 days of the child's birth.

6. Bereavement Leave: 3 days for immediate family (spouse, children, parents, siblings). Additional 2 days for travel if required.

7. Marriage Leave: 3 days, applicable once during employment tenure.

All leave balances are visible in the Employee Self-Service (ESS) portal under the 'My Leave' section. Leave balance resets every January 1st for casual and sick leave. HR helpline: hr@techcorp.in | 1800-XXX-HRHR."""
    },
    {
        "id": "doc_002",
        "topic": "Leave Application Process",
        "text": """How to Apply for Leave at TechCorp India:

Step 1 — Login to the ESS Portal at ess.techcorp.in using your employee ID and password.
Step 2 — Navigate to Leave Management > Apply Leave.
Step 3 — Select the leave type (Earned/Sick/Casual), enter dates, and add a reason.
Step 4 — Submit the application. Your reporting manager receives an automatic email for approval.

Notice Period Requirements:
- Casual Leave: Apply at least 1 working day in advance.
- Earned Leave: Apply at least 5 working days in advance for leaves up to 3 days; 10 working days in advance for leaves of 4 or more days.
- Emergency situations: Inform your manager by phone/message and apply via portal within 24 hours.

Approval Workflow:
- Manager approves or rejects within 2 working days.
- If no action is taken by the manager in 3 working days, leave is auto-escalated to the manager's manager.
- HR can override approvals in exceptional cases.

Leave Rejection:
- If leave is rejected, the employee receives an email with the reason.
- The employee may discuss with their manager or escalate to HR at grievance@techcorp.in.
- A rejected leave application does not deduct leave balance.

Half-Day Leave: Select 'Half Day' option in the portal. Specify First Half (9 AM–1 PM) or Second Half (2 PM–6 PM). Half-day leave deducts 0.5 days from the respective leave balance."""
    },
    {
        "id": "doc_003",
        "topic": "Payroll and Salary Structure",
        "text": """TechCorp India Payroll and Salary Information:

Pay Date: Salaries are credited on the last working day of every month. If the last working day falls on a bank holiday, salary is credited the previous working day.

Salary Components (CTC Breakdown):
- Basic Salary: 40% of CTC. This is the base for PF, gratuity, and leave encashment calculations.
- House Rent Allowance (HRA): 50% of Basic for metro cities (Mumbai, Delhi, Bengaluru, Chennai, Kolkata, Hyderabad); 40% of Basic for non-metro locations.
- Special Allowance: Remaining CTC after all other components. Fully taxable.
- Conveyance Allowance: Rs. 1,600 per month (tax-exempt up to Rs. 19,200 per year as per Income Tax Act).
- Medical Allowance: Rs. 1,250 per month (Rs. 15,000 per year, tax-exempt on submission of medical bills).

Deductions:
- Provident Fund (PF): 12% of Basic Salary deducted from employee; employer also contributes 12%.
- Professional Tax: As per state slab (e.g., Rs. 200/month in Karnataka).
- TDS (Tax Deducted at Source): Based on declared investments and tax regime chosen.
- ESI: Applicable if gross salary is below Rs. 21,000/month. Employee contributes 0.75%, employer 3.25%.

Payslip: Available in the ESS portal under Payroll > My Payslips by the 5th of the following month. Payslips are password-protected (password: first 4 letters of your name in lowercase + date of birth in DDMM format).

Salary Revision: Annual appraisal increments are effective from April 1st each year. Revised salary is reflected in the April payslip."""
    },
    {
        "id": "doc_004",
        "topic": "Reimbursements and Expense Claims",
        "text": """TechCorp India Reimbursement Policy:

1. Travel Reimbursement:
   - Local travel for official work: Actuals via cab/auto/metro, up to Rs. 2,000 per day.
   - Outstation travel: Booked through the corporate travel portal (travel.techcorp.in). Business class allowed only for roles of VP and above.
   - Personal vehicle usage: Rs. 8 per km for two-wheelers; Rs. 12 per km for four-wheelers (with proof of travel).

2. Food and Meal Allowance:
   - Working late (beyond 8 PM): Rs. 300 meal allowance per day (manager approval required).
   - Client entertainment: Up to Rs. 5,000 per occurrence with manager + finance approval and original bills.

3. Work From Home Allowance:
   - Internet: Rs. 500/month reimbursed on submission of bill.
   - Mobile: Rs. 300/month for employees in client-facing roles.

4. How to Claim:
   - Submit expense claims via ESS Portal > Reimbursements > New Claim within 30 days of expense.
   - Attach original bills/invoices (JPG, PDF, max 5 MB per file).
   - Claims above Rs. 5,000 require manager approval; above Rs. 20,000 require additional finance approval.
   - Approved claims are processed in the next payroll cycle (credited with salary).
   - Claims submitted after 30 days will NOT be processed without written HR approval.

5. Non-Reimbursable Items: Personal purchases, alcohol, fines/penalties, travel for personal reasons, gifts above Rs. 500."""
    },
    {
        "id": "doc_005",
        "topic": "Public Holidays and Company Holidays 2025",
        "text": """TechCorp India — Official Holiday List for 2025:

National Holidays (Mandatory — 3 days):
- January 26: Republic Day
- August 15: Independence Day
- October 2: Gandhi Jayanti

Company-Declared Holidays (Employees choose 8 from the list below based on their location/religion):
- January 1: New Year's Day
- January 14: Makar Sankranti / Pongal
- February 26: Maha Shivratri
- March 14: Holi
- March 31: Id-ul-Fitr (Eid)
- April 14: Dr. B.R. Ambedkar Jayanti / Tamil New Year
- April 18: Good Friday
- May 12: Buddha Purnima
- June 7: Id-ul-Zuha (Bakrid)
- August 16: Janmashtami
- October 2: Dussehra
- October 20: Diwali
- October 21: Diwali (second day)
- November 5: Guru Nanak Jayanti
- November 15: Milad-un-Nabi
- December 25: Christmas

How to Select Holidays:
- Log in to ESS Portal > Holiday Calendar > Select My Holidays.
- Selection must be done before January 15, 2025.
- Employees who miss the deadline are assigned the default company-wide 8 holidays.

Note: Working on a declared holiday earns compensatory off (comp off), which must be taken within 60 days. Comp off is applied via the ESS portal under Leave > Comp Off Request."""
    },
    {
        "id": "doc_006",
        "topic": "Onboarding Process for New Employees",
        "text": """TechCorp India New Employee Onboarding Guide:

Before Joining (Pre-Onboarding):
- Accept the offer letter via the link sent to your registered email within 5 days of receipt.
- Upload documents on the onboarding portal (onboard.techcorp.in): PAN card, Aadhaar card, last 3 months' salary slips, previous employer relieving letter, educational certificates, bank account details (cancelled cheque or passbook copy).
- If documents are pending, flag them to your HR SPOC before Day 1.

Day 1 Activities:
- Report to reception by 9:30 AM. Carry original documents for verification.
- Collect temporary access badge from the security desk.
- Attend HR Induction session (9:30 AM – 12:30 PM): Company overview, policies, culture, compliance, code of conduct.
- IT setup: Laptop, email ID, VPN access configured by IT helpdesk (ithelpdesk@techcorp.in | Ext: 1234).

First Week:
- Complete mandatory e-learning modules on the LMS portal (lms.techcorp.in): POSH, Data Privacy, Information Security, Code of Conduct.
- Meet your reporting manager and team.
- Set up your ESS portal access (ess.techcorp.in).

Probation Period:
- Duration: 6 months from the date of joining.
- Performance is reviewed at the 3-month and 6-month marks.
- Probation may be extended by up to 3 months based on performance review.
- Leave entitlement during probation: Sick and Casual leave only (Earned Leave accrual starts from Day 1 but can only be availed after probation confirmation).
- Notice period during probation: 1 month (from either side).

Employee ID Card: Permanent ID card issued within 7 working days of joining from Admin desk (Ground Floor)."""
    },
    {
        "id": "doc_007",
        "topic": "Work From Home Policy",
        "text": """TechCorp India Work From Home (WFH) Policy:

Eligibility:
- Employees who have completed their probation period (6 months) are eligible for WFH.
- Employees on probation are required to work from office 5 days a week.
- Roles marked as 'mandatory on-site' (e.g., data centre operations, facilities, reception) are not eligible for WFH.

WFH Entitlement:
- Standard employees: Up to 2 days WFH per week (Monday and Friday preferred).
- Employees with special approval (medical condition, childcare): Up to 4 days WFH per week.
- Hybrid schedule must be discussed and agreed upon with the reporting manager.

WFH Approval Process:
- Regular recurring WFH: Set up a 'WFH Schedule' in ESS Portal > Attendance > WFH Schedule. Requires manager approval once, valid for 3 months before renewal.
- Occasional WFH: Request via ESS Portal on the day before or morning of. Manager approves or declines within 2 hours.
- Emergency WFH (e.g., bad weather, illness): Inform manager on WhatsApp/phone and update portal within the same day.

WFH Expectations:
- Available on official communication channels (Teams/Slack) from 9 AM to 6 PM.
- Camera must be on during team meetings unless the meeting host has permitted otherwise.
- Attendance is marked via ESS portal log-in (geolocation disabled for WFH).
- Productivity metrics are tracked via manager assessment; consistent non-delivery on WFH days may lead to WFH privilege revocation.

Internet reimbursement of Rs. 500/month is available for WFH employees. Refer to the Reimbursements policy for details."""
    },
    {
        "id": "doc_008",
        "topic": "Performance Appraisal and Increments",
        "text": """TechCorp India Performance Appraisal Policy:

Appraisal Cycle:
- Annual appraisal: January to March each year (for the previous April–March performance year).
- Mid-year check-in: September (feedback only, no rating or increment).

Rating Scale:
- 5 – Outstanding: Consistently exceeds all expectations. Eligible for highest increment band and fast-track promotion.
- 4 – Exceeds Expectations: Regularly goes beyond role requirements.
- 3 – Meets Expectations: Fully delivers on role requirements. Standard increment.
- 2 – Partially Meets Expectations: Improvement plan (PIP) initiated. Minimum or no increment.
- 1 – Does Not Meet Expectations: PIP mandatory. Increment withheld.

Process:
1. Employee completes self-appraisal on ESS Portal > Performance > Self Appraisal (January 1–15).
2. Manager completes manager review and assigns rating (January 16–31).
3. Calibration session conducted across the team (February).
4. Final ratings communicated via ESS portal and manager 1:1 discussion (March 1–15).
5. Increment letters issued by March 31; effective April 1.

Increment Bands (approximate, subject to change):
- Rating 5: 15–25% increment
- Rating 4: 10–15% increment
- Rating 3: 6–10% increment
- Rating 2: 0–5% increment
- Rating 1: 0% increment

Promotion:
- Promotions are recommended by the manager during calibration.
- Minimum 2 years at current grade required for promotion eligibility.
- A rating of 4 or 5 in the current year is mandatory for promotion.

Grievance on Rating: Employees may raise a rating dispute with HR within 7 days of receiving their final rating via grievance@techcorp.in."""
    },
    {
        "id": "doc_009",
        "topic": "Grievance and Escalation Policy",
        "text": """TechCorp India Grievance Redressal Policy:

Types of Grievances Covered:
- Workplace harassment or discrimination (including POSH-related complaints)
- Unfair treatment by manager or colleagues
- Pay discrepancy or payroll errors
- Policy violations by the company or a team member
- Appraisal rating disputes
- Reimbursement delays or rejections

How to Raise a Grievance:
Step 1 — Informal Resolution: Speak directly with the concerned person or your reporting manager. Most issues are resolved at this stage.
Step 2 — Formal Grievance: If unresolved in 5 working days, submit a formal grievance:
   - Via ESS Portal: HR > Grievance > New Grievance
   - Via Email: grievance@techcorp.in (use your official email ID)
   - Grievance form must include: your employee ID, nature of grievance, details of incident, date, and any supporting documents.
Step 3 — HR Investigation: HR will acknowledge receipt within 2 working days and assign a case number. Investigation completed within 15 working days.
Step 4 — Resolution: HR communicates resolution in writing. Employee may appeal within 7 days if unsatisfied.
Step 5 — Escalation: If still unresolved, escalate to the HR Head at hrhead@techcorp.in.

POSH (Prevention of Sexual Harassment):
- TechCorp has an Internal Committee (IC) for POSH as per the POSH Act 2013.
- Complaints can be filed directly with IC at posh@techcorp.in.
- IC contact: Ms. Anita Sharma (IC Chairperson) | anita.sharma@techcorp.in.

Confidentiality: All grievances are handled with strict confidentiality. Retaliation against a complainant is a disciplinary offence.

HR Helpline: 1800-XXX-HRHR (Monday–Friday, 9 AM–6 PM) | hr@techcorp.in"""
    },
    {
        "id": "doc_010",
        "topic": "Exit and Resignation Process",
        "text": """TechCorp India Exit and Resignation Policy:

Resignation Process:
1. Submit resignation via ESS Portal > HR > Resignation Initiation, or send a formal email to your manager and hr@techcorp.in.
2. Notice period begins from the date HR officially acknowledges the resignation (within 1 working day).

Notice Period:
- During probation: 1 month
- Post-confirmation (up to 5 years tenure): 2 months
- Post-confirmation (above 5 years tenure): 3 months
- Notice period buyout: Allowed with manager and HR approval. Buyout amount = (Basic salary / 26) × remaining notice days.
- Early release: HR may waive notice period at company's discretion (e.g., if replacement is found earlier).

Exit Formalities (to complete before last working day):
- Return laptop, ID card, access badge, and any company assets to IT and Admin.
- Complete knowledge transfer with replacement/team (documented proof required).
- Clear all pending tasks or hand over to the manager.
- Complete exit interview with HR.

Full and Final (F&F) Settlement:
- Processed within 45 days of the last working day.
- Includes: Pending salary, leave encashment (Earned Leave balance), performance bonus (if applicable, pro-rated).
- Deductions: Notice period shortfall (if any), asset damage costs.
- F&F amount credited directly to the bank account on record.

Documents Issued After Relieving:
- Relieving Letter: Issued on the last working day (email copy); physical copy within 5 working days.
- Experience Letter: Issued within 7 working days of the last working day.
- Form 16: Issued by May 31st for the relevant financial year.

PF Withdrawal:
- Apply on the EPFO portal (epfindia.gov.in) using your UAN number.
- UAN is available in your payslip or from HR.
- PF can be transferred to new employer PF account instead of withdrawal.

Rehire Policy: Former employees in good standing are eligible for rehire after a minimum gap of 6 months."""
    }
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6745.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Knowledge base ready: 10 documents
   • Leave Policy Overview
   • Leave Application Process
   • Payroll and Salary Structure
   • Reimbursements and Expense Claims
   • Public Holidays and Company Holidays 2025
   • Onboarding Process for New Employees
   • Work From Home Policy
   • Performance Appraisal and Increments
   • Grievance and Escalation Policy
   • Exit and Resignation Process


In [9]:
# ── Test retrieval before building the graph ──────────────
test_query = "How many casual leaves do I get per year?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: How many casual leaves do I get per year?

Top 3 retrieved chunks:

[1] Topic: Leave Policy Overview
    Text: TechCorp India offers its employees the following types of leave annually:

1. Earned Leave (EL): 18 days per year, accrued at 1.5 days per month. Unused EL can be carried forward up to a maximum of 3...

[2] Topic: Payroll and Salary Structure
    Text: TechCorp India Payroll and Salary Information:

Pay Date: Salaries are credited on the last working day of every month. If the last working day falls on a bank holiday, salary is credited the previous...

[3] Topic: Public Holidays and Company Holidays 2025
    Text: TechCorp India — Official Holiday List for 2025:

National Holidays (Mandatory — 3 days):
- January 26: Republic Day
- August 15: Independence Day
- October 2: Gandhi Jayanti

Company-Declared Holiday...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

In [12]:
from typing import TypedDict, List

class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question
    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history
    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"
    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names
    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from web search tool
    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response
    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter
    # ── HR-specific fields ─────────────────────────────────
    employee_name: str          # extracted from conversation if user shares it

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from web search tool

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter

    # ── HR-specific fields ─────────────────────────────────
    employee_name: str          # extracted from conversation if user shares it

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'employee_name']
State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'employee_name']


---
## Part 3 — Node Functions

In [13]:
# ── Node 1: Memory ─────────────────────────────────────────
def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]

    # Extract employee name if user shares it
    employee_name = state.get("employee_name", "")
    q_lower = state["question"].lower()
    if "my name is" in q_lower:
        parts = q_lower.split("my name is")
        if len(parts) > 1:
            employee_name = parts[1].strip().split()[0].capitalize()

    return {"messages": msgs, "employee_name": employee_name}


# Quick test
test_state = {"question": "My name is Rahul. What are my leave options?", "messages": [], "employee_name": ""}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print(f"employee_name extracted: {result['employee_name']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'My name is Rahul. What are my leave options?'}]
employee_name extracted: Rahul.
✅ memory_node works


In [14]:
# ── Node 2: Router ─────────────────────────────────────────
def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for an HR Policy chatbot for a company called TechCorp India.

Available options:
- retrieve: search the knowledge base for HR policy information (leave, payroll, reimbursements, onboarding, appraisal, WFH, grievance, exit)
- memory_only: answer from conversation history (e.g. 'what did you just say?', 'can you repeat that?', 'tell me more about what you said')
- tool: use the web search tool for live external data (e.g. current PF rates, government holiday list, labour law updates, statutory compliance questions)

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:       decision = "memory_only"
    elif "tool" in decision:       decision = "tool"
    else:                          decision = "retrieve"

    print(f"  [router] Route: {decision}")
    return {"route": decision}


# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

  [router] Route: memory_only
router_node test: route='memory_only' (expected: memory_only)


In [16]:
# ── Node 3: Retrieval ──────────────────────────────────────
def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "How many earned leaves do I get?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Leave Policy Overview', 'Payroll and Salary Structure', 'Reimbursements and Expense Claims']
  Context preview: [Leave Policy Overview]
TechCorp India offers its employees the following types of leave annually:

1. Earned Leave (EL): 18 days per year, accrued at 1.5 days per month. Unused EL can be carried forw...
✅ retrieval_node works


In [17]:
# ── Node 4: Tool — Web Search (DuckDuckGo) ─────────────────
def tool_node(state: CapstoneState) -> dict:
    """Web search for live HR/labour law data not in the knowledge base."""
    question = state["question"]
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(
                question + " India HR labour law 2025",
                max_results=3
            ))
        if results:
            tool_result = "\n\n".join(
                f"Source: {r['title']}\n{r['body'][:400]}"
                for r in results
            )
        else:
            tool_result = "No search results found for this query."
    except Exception as e:
        tool_result = f"Web search unavailable: {str(e)}. Please contact HR directly at hr@techcorp.in."

    return {"tool_result": tool_result}


print("✅ tool_node defined — uses DuckDuckGo web search")

✅ tool_node defined — uses DuckDuckGo web search


In [18]:
# ── Node 5: Answer ─────────────────────────────────────────
def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)
    emp_name     = state.get("employee_name", "")

    greeting = f" {emp_name}" if emp_name else ""

    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"WEB SEARCH RESULTS:\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = f"""You are HRBot, the friendly and knowledgeable HR Policy Assistant for TechCorp India.
Your job is to help employees understand company HR policies clearly and accurately.

STRICT RULES:
1. Answer using ONLY the information provided in the context below.
2. If the answer is not in the context, say exactly: "I don't have that information in my knowledge base. Please contact HR at hr@techcorp.in or call 1800-XXX-HRHR for assistance."
3. Do NOT add information from your training data or make up policy details.
4. Be concise, friendly, and use bullet points where helpful.
5. Always mention the relevant portal or contact if applicable.
{f'6. Address the employee as{greeting} where natural.' if emp_name else ''}

CONTEXT:
{context}"""
    else:
        system_content = """You are HRBot, the HR Policy Assistant for TechCorp India.
Answer based on the conversation history. If you lack information, say so and direct to HR."""

    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above. Do not add anything beyond what is written there."

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}


print("✅ answer_node defined")

✅ answer_node defined


In [19]:
# ── Node 6: Eval ───────────────────────────────────────────
FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer  = state.get("answer", "")
    context = state.get("retrieved", "")[:500]
    retries = state.get("eval_retries", 0)

    if not context:
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save ───────────────────────────────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}


print("✅ eval_node and save_node defined")

✅ eval_node and save_node defined


---
## Part 4 — Graph Assembly

In [21]:
# ── Routing functions ──────────────────────────────────────
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
def route_decision(state: CapstoneState) -> str:
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CapstoneState)

graph.add_node("memory",   memory_node)
graph.add_node("router",   router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip",     skip_retrieval_node)
graph.add_node("tool",     tool_node)
graph.add_node("answer",   answer_node)
graph.add_node("eval",     eval_node)
graph.add_node("save",     save_node)

graph.set_entry_point("memory")
graph.add_edge("memory", "router")

graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")
graph.add_edge("answer",   "eval")

graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes: memory → router → [retrieve/skip/tool] → answer → eval → save → END")

✅ Graph compiled successfully!
Nodes: memory → router → [retrieve/skip/tool] → answer → eval → save → END


---
## Part 5 — Testing

In [22]:
def ask(question: str, thread_id: str = "test") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    {"q": "How many casual leaves do I get per year?",                         "expect": "Should say 8 CL per year",                   "red_team": False},
    {"q": "When is my salary credited each month?",                            "expect": "Last working day of the month",               "red_team": False},
    {"q": "How do I apply for earned leave?",                                  "expect": "ESS portal steps",                           "red_team": False},
    {"q": "What is the WFH policy after probation?",                           "expect": "2 days per week",                            "red_team": False},
    {"q": "How do I claim internet reimbursement while working from home?",    "expect": "Rs 500/month via ESS portal",                "red_team": False},
    {"q": "What happens to my earned leave balance when I resign?",            "expect": "Encashed in F&F settlement",                 "red_team": False},
    {"q": "How does the appraisal process work?",                              "expect": "Self-appraisal Jan, manager review, increment Apr", "red_team": False},
    {"q": "What is the notice period for someone with 4 years tenure?",        "expect": "2 months",                                  "red_team": False},
    # Memory test — must reference prior conversation context
    {"q": "Can you repeat what you said about the notice period?",             "expect": "Should recall previous answer about notice period", "red_team": False},
    # Red-team: out-of-scope
    {"q": "What is the stock price of TechCorp today?",                        "expect": "Should admit it doesn't know",               "red_team": True},
    # Red-team: false premise
    {"q": "I heard employees get 30 casual leaves a year. Is that correct?",   "expect": "Should correct — it's 8 CL, not 30",        "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 11 test questions (2 red-team)


In [27]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
test_results = []

print("=" * 65)
print("RUNNING TEST SUITE — HR Policy Bot")
print("=" * 65)

# Use one thread for memory test (Q8-Q9 share thread)
memory_thread = "memory-test-thread"

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    # Use shared thread for questions 8 and 9 (memory test)
    tid = memory_thread if i in [7, 8] else f"test-{i}"
    result = ask(test["q"], thread_id=tid)

    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:250]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    passed = len(answer) > 20 and "error" not in answer.lower()[:50]
    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")

    test_results.append({"q": test["q"][:55], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*65}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

RUNNING TEST SUITE — HR Policy Bot

--- Test 1  ---
Q: How many casual leaves do I get per year?
  [router] Route: retrieve
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
A: You get 8 casual leaves per year. 

* Note: These leaves cannot be combined with Earned Leave, and a maximum of 3 consecutive casual leave days are allowed. Unused casual leaves lapse on December 31st. You can check your leave balance in the Employee
Route: retrieve | Faithfulness: 0.00
Expected: Should say 8 CL per year
Result: ✅ PASS

--- Test 2  ---
Q: When is my salary credited each month?
  [router] Route: retrieve
  [eval] Faithfulness: 1.00 ✅
A: Your salary is credited on the last working day of every month. If the last working day falls on a bank holiday, your salary is credited the previous working day.
Route: retrieve | Faithfulness: 1.00
Expected: Last working day of the month
Result: ✅ PASS

--- Test 3  ---
Q: How do I apply for earned leave?
  [router] Route: retrieve
  [eval] Faithfulne

---
## Part 6 — RAGAS Baseline Evaluation

In [28]:
RAGAS_QUESTIONS = [
    {
        "question": "How many earned leaves do employees get per year?",
        "ground_truth": "Employees get 18 earned leaves per year, accrued at 1.5 days per month. Unused EL can be carried forward up to 30 days."
    },
    {
        "question": "What are the salary components at TechCorp?",
        "ground_truth": "Salary components include Basic (40% of CTC), HRA (50% of Basic in metros, 40% in non-metros), Special Allowance, Conveyance Allowance (Rs 1600/month), and Medical Allowance (Rs 1250/month)."
    },
    {
        "question": "How many WFH days are allowed per week?",
        "ground_truth": "Standard employees can work from home up to 2 days per week after completing probation. Employees with special approval (medical/childcare) can WFH up to 4 days per week."
    },
    {
        "question": "What is the notice period for an employee with 3 years of experience?",
        "ground_truth": "An employee with 3 years of service (post-confirmation, under 5 years) needs to serve a 2-month notice period."
    },
    {
        "question": "How do I raise a grievance at TechCorp?",
        "ground_truth": "First try informal resolution with your manager. If unresolved in 5 working days, raise a formal grievance via ESS Portal under HR > Grievance or email grievance@techcorp.in. HR acknowledges within 2 working days and resolves within 15 working days."
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:60]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [router] Route: retrieve
  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 1.00 ✅
  ✓ How many earned leaves do employees get per year?
  [router] Route: retrieve
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
  ✓ What are the salary components at TechCorp?
  [router] Route: retrieve
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
  ✓ How many WFH days are allowed per week?
  [router] Route: retrieve
  [eval] Faithfulness: 0.00 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
  ✓ What is the notice period for an employee with 3 years of ex
  [router] Route: retrieve
  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 0.50 ⚠️
  ✓ How do I raise a grievance at TechCorp?

✅ Eval dataset built: 5 rows


In [36]:
!pip install langchain-huggingface -q
!pip install ragas datasets


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset

try:
    # Configure RAGAS to use Groq instead of OpenAI
    ragas_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.3-70b-versatile", temperature=0))
    ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

    for metric in [faithfulness, answer_relevancy, context_precision]:
        metric.llm = ragas_llm
        metric.embeddings = ragas_emb

    ragas_data = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores in your written summary below.")

except Exception as e:
    print(f"RAGAS evaluation failed ({e}) — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:50]:50s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_31432\3642538992.py:11: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatGroq(model="llama-3.3-70b-versatile", temperature=0))
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6457.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_31432\3642538992.py:12: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding provide

Running RAGAS evaluation (1-2 minutes)...
RAGAS evaluation failed (All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]) — running manual faithfulness scoring
  Q: How many earned leaves do employees get per year?  → 0.00
  Q: What are the salary components at TechCorp?        → 0.00
  Q: How many WFH days are allowed per week?            → 0.80
  Q: What is the notice period for an employee with 3 y → 0.00
  Q: How do I raise a grievance at TechCorp?            → 0.00

Baseline faithfulness: 0.160
Install RAGAS for full evaluation: pip install ragas datasets


---
## Part 7 — Deployment (Streamlit App)

Run the cell below to write `capstone_streamlit.py`, then run it with:
```
streamlit run capstone_streamlit.py
```

In [38]:
streamlit_code = '''
"""
capstone_streamlit.py — TechCorp HR Policy Bot
Run: streamlit run capstone_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(
    page_title="TechCorp HR Policy Bot",
    page_icon="👔",
    layout="centered"
)

st.title("👔 TechCorp HR Policy Bot")
st.caption("Your 24/7 assistant for HR policies — leave, payroll, WFH, appraisal, and more.")

@st.cache_resource
def load_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    DOCUMENTS = [
        {"id":"doc_001","topic":"Leave Policy Overview","text":"""TechCorp India offers employees: Earned Leave (EL): 18 days/year accrued at 1.5 days/month, carry-forward up to 30 days. Sick Leave (SL): 12 days/year, no carry-forward, medical cert needed after 3 days. Casual Leave (CL): 8 days/year, max 3 consecutive days, no carry-forward. Maternity Leave: 26 weeks fully paid (first 2 children); 12 weeks for 3rd child onwards. Paternity Leave: 5 working days within 30 days of child birth. Bereavement Leave: 3 days for immediate family. Marriage Leave: 3 days (once per tenure). HR helpline: hr@techcorp.in | 1800-XXX-HRHR."""},
        {"id":"doc_002","topic":"Leave Application Process","text":"""Apply via ESS portal (ess.techcorp.in) > Leave Management > Apply Leave. Notice: CL — 1 day advance; EL up to 3 days — 5 days advance; EL 4+ days — 10 days advance. Manager approves within 2 working days; auto-escalates at 3 days. Rejection: email with reason sent; escalate to HR at grievance@techcorp.in. Half-day leave deducts 0.5 days."""},
        {"id":"doc_003","topic":"Payroll and Salary Structure","text":"""Salary credited on last working day of month. Components: Basic (40% CTC), HRA (50% Basic metro / 40% non-metro), Special Allowance, Conveyance Rs 1600/month, Medical Rs 1250/month. Deductions: PF 12% of Basic, Professional Tax per state, TDS, ESI if gross < Rs 21000. Payslip on ESS portal by 5th of next month. Annual increment effective April 1."""},
        {"id":"doc_004","topic":"Reimbursements and Expense Claims","text":"""Travel: local actuals up to Rs 2000/day; outstation via corporate portal. Personal vehicle: Rs 8/km (2-wheeler), Rs 12/km (4-wheeler). Late work meal: Rs 300 (after 8 PM, manager approval). Internet WFH: Rs 500/month. Mobile: Rs 300/month (client-facing). Submit via ESS > Reimbursements within 30 days with receipts. Claims above Rs 5000 need manager approval. Processed next payroll cycle."""},
        {"id":"doc_005","topic":"Public Holidays and Company Holidays 2025","text":"""National holidays (mandatory): Jan 26 Republic Day, Aug 15 Independence Day, Oct 2 Gandhi Jayanti. Company holidays (choose 8): Jan 1, Jan 14 Pongal, Mar 14 Holi, Mar 31 Eid, Apr 18 Good Friday, Oct 20-21 Diwali, Dec 25 Christmas, and others. Select holidays in ESS portal by Jan 15. Working on declared holiday earns comp-off (must be taken within 60 days)."""},
        {"id":"doc_006","topic":"Onboarding Process for New Employees","text":"""Upload documents at onboard.techcorp.in before Day 1: PAN, Aadhaar, salary slips, relieving letter, certificates, bank details. Day 1: Report 9:30 AM, HR induction, IT setup (ithelpdesk@techcorp.in | Ext 1234). Week 1: Complete LMS modules (POSH, Data Privacy, Code of Conduct). Probation: 6 months; EL accrues but available only after confirmation. Notice during probation: 1 month. ID card issued within 7 working days."""},
        {"id":"doc_007","topic":"Work From Home Policy","text":"""Eligible after 6-month probation. Standard employees: up to 2 WFH days/week. Special approval (medical/childcare): up to 4 days/week. Set recurring WFH schedule in ESS > Attendance > WFH Schedule (valid 3 months). Be available on Teams/Slack 9 AM-6 PM. Camera on in team meetings. Attendance marked via ESS login. Internet reimbursement Rs 500/month available."""},
        {"id":"doc_008","topic":"Performance Appraisal and Increments","text":"""Annual appraisal: Jan-Mar for Apr-Mar performance year. Ratings: 5 Outstanding (15-25% increment), 4 Exceeds (10-15%), 3 Meets (6-10%), 2 Partially Meets (0-5%), 1 Does Not Meet (0%). Process: Self-appraisal Jan 1-15, manager review Jan 16-31, calibration Feb, results Mar 1-15, increment letters by Mar 31 effective Apr 1. Promotion needs 2+ years at grade, rating 4 or 5. Dispute within 7 days to grievance@techcorp.in."""},
        {"id":"doc_009","topic":"Grievance and Escalation Policy","text":"""Grievances: harassment, unfair treatment, pay discrepancy, policy violations, appraisal disputes, reimbursement issues. Step 1: Informal resolution with manager (5 days). Step 2: Formal grievance via ESS > HR > Grievance or grievance@techcorp.in. Step 3: HR investigates within 15 working days. Step 4: Resolution communicated; appeal within 7 days. Step 5: Escalate to hrhead@techcorp.in. POSH: posh@techcorp.in | IC Chairperson Anita Sharma. HR Helpline: 1800-XXX-HRHR."""},
        {"id":"doc_010","topic":"Exit and Resignation Process","text":"""Resign via ESS > HR > Resignation or email manager + hr@techcorp.in. Notice period: probation 1 month, post-confirmation up to 5 years 2 months, above 5 years 3 months. Notice buyout allowed with approval. Exit formalities: return assets, knowledge transfer, exit interview. F&F settlement within 45 days (pending salary + EL encashment). Relieving letter on last day; experience letter within 7 days. PF withdrawal via epfindia.gov.in using UAN from payslip.""},
    ]

    client = chromadb.Client()
    try: client.delete_collection("capstone_kb")
    except: pass
    collection = client.create_collection("capstone_kb")
    texts = [d["text"] for d in DOCUMENTS]
    collection.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
    )

    class CapstoneState(TypedDict):
        question:      str
        messages:      List[dict]
        route:         str
        retrieved:     str
        sources:       List[str]
        tool_result:   str
        answer:        str
        faithfulness:  float
        eval_retries:  int
        employee_name: str

    FAITHFULNESS_THRESHOLD = 0.7
    MAX_EVAL_RETRIES = 2

    def memory_node(state):
        msgs = state.get("messages", [])
        msgs = msgs + [{"role": "user", "content": state["question"]}]
        if len(msgs) > 6: msgs = msgs[-6:]
        emp = state.get("employee_name", "")
        q = state["question"].lower()
        if "my name is" in q:
            parts = q.split("my name is")
            if len(parts) > 1: emp = parts[1].strip().split()[0].capitalize()
        return {"messages": msgs, "employee_name": emp}

    def router_node(state):
        question = state["question"]
        messages = state.get("messages", [])
        recent = "; ".join(f"{m[\x27role\x27]}: {m[\x27content\x27][:60]}" for m in messages[-3:-1]) or "none"
        prompt = f"""You are a router for an HR Policy chatbot for TechCorp India.
Options:
- retrieve: search knowledge base for HR policies (leave, payroll, reimbursements, onboarding, appraisal, WFH, grievance, exit)
- memory_only: answer from conversation history
- tool: use web search for live data (PF rates, govt holidays, labour law updates)
Recent: {recent}
Question: {question}
Reply with ONLY one word: retrieve / memory_only / tool"""
        response = llm.invoke(prompt)
        decision = response.content.strip().lower()
        if "memory" in decision: decision = "memory_only"
        elif "tool" in decision: decision = "tool"
        else: decision = "retrieve"
        return {"route": decision}

    def retrieval_node(state):
        q_emb = embedder.encode([state["question"]]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=3)
        chunks = results["documents"][0]
        topics = [m["topic"] for m in results["metadatas"][0]]
        context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
        return {"retrieved": context, "sources": topics}

    def skip_retrieval_node(state):
        return {"retrieved": "", "sources": []}

    def tool_node(state):
        question = state["question"]
        try:
            from ddgs import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(question + " India HR labour law 2025", max_results=3))
            tool_result = "\n\n".join(f"Source: {r[\x27title\x27]}\n{r[\x27body\x27][:400]}" for r in results) if results else "No results found."
        except Exception as e:
            tool_result = f"Web search unavailable: {str(e)}. Contact HR at hr@techcorp.in."
        return {"tool_result": tool_result}

    def answer_node(state):
        question = state["question"]
        retrieved = state.get("retrieved", "")
        tool_result = state.get("tool_result", "")
        messages = state.get("messages", [])
        eval_retries = state.get("eval_retries", 0)
        emp_name = state.get("employee_name", "")
        greeting = f" {emp_name}" if emp_name else ""
        context_parts = []
        if retrieved: context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
        if tool_result: context_parts.append(f"WEB SEARCH:\n{tool_result}")
        context = "\n\n".join(context_parts)
        if context:
            system_content = f"""You are HRBot, the friendly HR Policy Assistant for TechCorp India.
Answer using ONLY the information in the context below.
If not found, say: I don't have that information. Please contact HR at hr@techcorp.in or call 1800-XXX-HRHR.
Do NOT add information from your training data.{f\x27 Address employee as{greeting}.\x27 if emp_name else \x27\x27}

CONTEXT:\n{context}"""
        else:
            system_content = "You are HRBot for TechCorp India. Answer from conversation history or direct to HR."
        if eval_retries > 0:
            system_content += "\n\nIMPORTANT: Answer using ONLY information explicitly in the context above."
        lc_msgs = [SystemMessage(content=system_content)]
        for msg in messages[:-1]:
            lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user" else AIMessage(content=msg["content"]))
        lc_msgs.append(HumanMessage(content=question))
        response = llm.invoke(lc_msgs)
        return {"answer": response.content}

    def eval_node(state):
        answer = state.get("answer", "")
        context = state.get("retrieved", "")[:500]
        retries = state.get("eval_retries", 0)
        if not context:
            return {"faithfulness": 1.0, "eval_retries": retries + 1}
        prompt = f"""Rate faithfulness 0.0-1.0. Reply ONLY with a number.
1.0=fully faithful, 0.5=some hallucination, 0.0=mostly hallucinated.
Context: {context}
Answer: {answer[:300]}"""
        result = llm.invoke(prompt).content.strip()
        try:
            score = float(result.split()[0].replace(",", "."))
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        return {"faithfulness": score, "eval_retries": retries + 1}

    def save_node(state):
        messages = state.get("messages", [])
        messages = messages + [{"role": "assistant", "content": state["answer"]}]
        return {"messages": messages}

    def route_decision(state):
        r = state.get("route", "retrieve")
        if r == "tool": return "tool"
        if r == "memory_only": return "skip"
        return "retrieve"

    def eval_decision(state):
        score = state.get("faithfulness", 1.0)
        retries = state.get("eval_retries", 0)
        if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
            return "save"
        return "answer"

    graph = StateGraph(CapstoneState)
    graph.add_node("memory", memory_node)
    graph.add_node("router", router_node)
    graph.add_node("retrieve", retrieval_node)
    graph.add_node("skip", skip_retrieval_node)
    graph.add_node("tool", tool_node)
    graph.add_node("answer", answer_node)
    graph.add_node("eval", eval_node)
    graph.add_node("save", save_node)
    graph.set_entry_point("memory")
    graph.add_edge("memory", "router")
    graph.add_conditional_edges("router", route_decision, {"retrieve": "retrieve", "skip": "skip", "tool": "tool"})
    graph.add_edge("retrieve", "answer")
    graph.add_edge("skip", "answer")
    graph.add_edge("tool", "answer")
    graph.add_edge("answer", "eval")
    graph.add_conditional_edges("eval", eval_decision, {"answer": "answer", "save": "save"})
    graph.add_edge("save", END)
    agent_app = graph.compile(checkpointer=MemorySaver())
    return agent_app, embedder, collection, DOCUMENTS


try:
    agent_app, embedder, collection, DOCUMENTS = load_agent()
    st.success(f"✅ Knowledge base loaded — {collection.count()} HR policy documents")
except Exception as e:
    st.error(f"Failed to load agent: {e}")
    st.stop()

# ── Session state ─────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("👔 TechCorp HR Bot")
    st.write("Your 24/7 HR policy assistant. Ask anything about company policies.")
    st.divider()
    st.write("**Topics I can help with:**")
    topics = [d["topic"] for d in DOCUMENTS]
    for t in topics:
        st.write(f"• {t}")
    st.divider()
    st.write(f"🔗 Session ID: `{st.session_state.thread_id}`")
    st.write("📞 HR Helpline: 1800-XXX-HRHR")
    st.write("📧 hr@techcorp.in")
    if st.button("🗑️ New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

# ── Display history ───────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask an HR policy question..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        with st.spinner("Checking HR policies..."):
            config = {"configurable": {"thread_id": st.session_state.thread_id}}
            result = agent_app.invoke({"question": prompt}, config=config)
            answer = result.get("answer", "Sorry, I could not generate an answer. Please contact hr@techcorp.in.")
        st.write(answer)
        faith = result.get("faithfulness", 0.0)
        sources = result.get("sources", [])
        if faith > 0:
            st.caption(f"Faithfulness score: {faith:.2f} | Sources: {sources}")

    st.session_state.messages.append({"role": "assistant", "content": answer})

# ── Footer disclaimer ─────────────────────────────────────
st.divider()
st.caption("⚠️ This bot provides policy information only. For official HR decisions, please contact HR directly.")
'''

with open("capstone_streamlit.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("✅ capstone_streamlit.py written successfully")
print("")
print("To run: streamlit run capstone_streamlit.py")

✅ capstone_streamlit.py written successfully

To run: streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary

## My Capstone Summary

**Name:** [AKKSHITA ISA]

**Roll Number:** [23052055]

**Batch/Program:** Agentic AI Hands-On Course 2026

**Domain chosen:** HR Policy Bot

**What the agent does:** HRBot is a 24/7 intelligent HR Policy Assistant for TechCorp India. It helps employees instantly get accurate answers about leave policies, payroll, reimbursements, WFH rules, performance appraisals, grievances, and the exit process — without needing to contact HR directly. It uses a ChromaDB knowledge base of 10 HR policy documents, maintains conversation memory across turns, and uses web search to fetch live government holiday lists and statutory labour law updates.

**Knowledge base:** 10 documents covering: Leave Policy Overview, Leave Application Process, Payroll and Salary Structure, Reimbursements and Expense Claims, Public and Company Holidays 2025, Onboarding Process, Work From Home Policy, Performance Appraisal and Increments, Grievance and Escalation Policy, and Exit and Resignation Process.

**Tool used:** Web search via DuckDuckGo (ddgs). Used when employees ask about live data not in the static KB — e.g., current PF contribution rates, government holiday declarations, or latest labour law statutory amendments. The router correctly routes these queries to the tool instead of the knowledge base.

**RAGAS baseline scores:**
- Faithfulness: [Fill after running]
- Answer Relevance: [Fill after running]
- Context Precision: [Fill after running]

**Test results:** [X] / 11 tests passed. Red-team: [X] / 2 passed.

**One thing I would improve with more time:** I would replace the hand-written policy summaries with actual PDF uploads of the company HR handbook, using a PDF loader to split and embed real policy documents. This would make the knowledge base more accurate and comprehensive, and improve context precision scores in RAGAS evaluation.

**Most surprising thing I learned building this:** The router prompt quality has an outsized impact on the entire agent's performance. A poorly written router prompt causes misrouting, which no amount of improvement to the answer node can fix — because the agent never even retrieves the right context.


---
## Submission Checklist

- [x] All TODO sections in the notebook have been filled in
- [x] Knowledge base has 10 documents covering all major HR policy areas
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 11 questions
- [ ] RAGAS baseline scores are recorded in the written summary
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [x] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py`
3. `agent.py` (shared agent module)
